# Notebook 2: Machine Architecture & Data Representation
### COMP 1150 — Computer Science Concepts
**Brendan Shea, PhD**

📺 **Lecture video:** *(link coming soon)*

## Learning Outcomes

By the end of this notebook, you will be able to:

- **Explain** how a transistor becomes a logic gate, and how logic gates combine to do arithmetic
- **Describe** the von Neumann architecture, including the control unit, ALU, registers, and the fetch–decode–execute cycle
- **Explain** the memory hierarchy and why caching exists
- **Convert** numbers between binary, decimal, and hexadecimal by hand
- **Represent** negative numbers using two's complement, and decode a two's-complement byte
- **Explain** how text, images, and sound are all stored as numbers
- **Identify** how a representation choice (like a fixed-size integer) can cause real bugs

*Maps to course LOs: 1 (machine architecture, data representation, processor–memory interaction) and 2 (data storage, number systems, binary)*

## A Memory Budget Crisis

Welcome to **Eight Bits & Bob**, a small game studio shipping its next title onto the **PixelBox 8** — an 8-bit console that, per the box, comes "with up to four colours, two of which are brown."

The game is *Quest for the Reasonably Priced Sword*. The studio director, **Vesper Crunch** — who measures everything in bytes, including lunch — has just delivered the bad news: the entire game must fit in a memory budget smaller than a single modern photo. Every character, every sound, every number on screen has to be squeezed into 0s and 1s, efficiently.

That constraint is a gift. It forces us to answer a question most programmers never think about: **when everything is just 0s and 1s, how do you store *anything*?** A score? A minus sign? A dragon? A trumpet? And once it's stored, **how does the machine actually compute with it?**

This notebook answers both, from the transistor up.

## The Roadmap

We'll build understanding bottom-up — the same direction a computer is physically built:

| Part | Question it answers |
|---|---|
| Transistors → gates → arithmetic | How does a chip *do* anything? |
| Inside the machine | How are the CPU, memory, and I/O actually wired and used? |
| The memory hierarchy | Why are there so many *kinds* of memory? |
| Consoles as a fossil record | How has all of this changed over 45 years? |
| Binary & hexadecimal | How do we write numbers with only 0s and 1s? |
| Two's complement | How do we store *negative* numbers? |
| Text, images, sound | How is everything else stored as numbers? |
| When representation breaks | Why does this matter for real bugs? |

Several parts end with a hands-on activity — including **practice games built into this notebook** that you can replay before a quiz.

Notebook 1 closed with a promise: *we'd see how a transistor becomes a logic gate, a logic gate becomes an adder, and an adder becomes a computer.* Let's pay it off.

---
## From Transistor to Arithmetic

**Dot Mainframe**, the studio's hardware engineer — who named her cat "Cache" — starts at the bottom.

A **transistor** is a tiny electrical switch with no moving parts. It has two states: current flows (call it **1**) or it doesn't (call it **0**). That's it. That's the whole physical foundation of computing. A modern chip has *billions* of these switches; the PixelBox 8 has a few thousand.

One switch isn't interesting. The magic starts when you wire a few together so their combined behavior follows a *rule*. A small bundle of transistors wired to follow one logical rule is called a **logic gate**.

In [ ]:
# The four gates you must know. Each takes inputs (0 or 1) and produces one output.
from graphviz import Digraph

g = Digraph(graph_attr={"rankdir": "LR"})
g.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

for name in ["AND", "OR", "XOR"]:
    g.node(f"{name}_A", "A", shape="circle", fillcolor="lightgreen")
    g.node(f"{name}_B", "B", shape="circle", fillcolor="lightgreen")
    g.node(f"{name}_G", f"{name}\ngate")
    g.node(f"{name}_Y", "output", shape="circle", fillcolor="lightcoral")
    g.edge(f"{name}_A", f"{name}_G")
    g.edge(f"{name}_B", f"{name}_G")
    g.edge(f"{name}_G", f"{name}_Y")

g.node("NOT_A", "A", shape="circle", fillcolor="lightgreen")
g.node("NOT_G", "NOT\ngate")
g.node("NOT_Y", "output", shape="circle", fillcolor="lightcoral")
g.edge("NOT_A", "NOT_G")
g.edge("NOT_G", "NOT_Y")

g

### How Each Gate Behaves

A gate's rule is captured in a **truth table** — every possible input, and the output it produces. There are no surprises and no choices: same inputs, same output, every time.

**AND** — output 1 only if *both* inputs are 1:

| A | B | A AND B |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | **1** |

**OR** — output 1 if *either* input is 1. &nbsp;&nbsp; **XOR** ("exclusive or") — output 1 if the inputs are *different*. &nbsp;&nbsp; **NOT** — one input, flipped.

| A | B | A OR B | A XOR B |&nbsp;&nbsp;|&nbsp;A&nbsp;| NOT A |
|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 |&nbsp;&nbsp;| 0 | 1 |
| 0 | 1 | 1 | 1 |&nbsp;&nbsp;| 1 | 0 |
| 1 | 0 | 1 | 1 |&nbsp;&nbsp;|&nbsp;|&nbsp;|
| 1 | 1 | 1 | 0 |&nbsp;&nbsp;|&nbsp;|&nbsp;|

Memorize these four. Every calculation a computer has ever done is built from them.

### From Gates to Addition

**Junior Dev Tobble** — who has never seen a number bigger than 255 and would like to keep it that way — asks the obvious question: *how does a pile of gates actually add 1 + 1?*

Look at the XOR and AND results for the inputs `1` and `1`:

- `1 XOR 1 = 0` → that's the **sum digit** (1 + 1 in binary is `10`, so the digit you write down is 0)
- `1 AND 1 = 1` → that's the **carry digit** (the 1 you carry into the next column)

So an XOR gate and an AND gate, side by side, *add two bits*: XOR gives the result digit, AND gives the carry. This pairing is called a **half-adder**. You don't need to build one — just hold onto the idea: **arithmetic is just gates wired cleverly.**

In [ ]:
# Conceptual view of a half-adder: the same two input bits feed two gates.
ha = Digraph(graph_attr={"rankdir": "LR"})
ha.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

ha.node("A", "bit A", shape="circle", fillcolor="lightgreen")
ha.node("B", "bit B", shape="circle", fillcolor="lightgreen")
ha.node("X", "XOR")
ha.node("N", "AND")
ha.node("S", "SUM digit",   shape="circle", fillcolor="lightcoral")
ha.node("C", "CARRY digit", shape="circle", fillcolor="lightcoral")

ha.edge("A", "X"); ha.edge("B", "X"); ha.edge("X", "S")
ha.edge("A", "N"); ha.edge("B", "N"); ha.edge("N", "C")

ha

Chain eight half-adders together (passing each carry into the next column) and you can add two 8-bit numbers. That chain is the core of the **ALU** — the **Arithmetic Logic Unit**, the part of the chip that does math and comparisons. Add some gates that select *which* operation to run, and you have, essentially, a processor's calculator.

That's the whole bottom of the stack: **transistor → gate → half-adder → ALU.** Next we'll see where the ALU sits inside a complete machine.

Before we go up a level, here's a practice game for gates. Run the cell, read the walk-through, then do the exercise underneath it.

In [ ]:
import random

def gate_arcade(rounds=5):
    """Random logic-gate practice. Type 0 or 1 as your answer."""
    gates = {
        "AND": lambda a, b: a & b,
        "OR":  lambda a, b: a | b,
        "XOR": lambda a, b: a ^ b,
    }
    score = 0
    for r in range(1, rounds + 1):
        name = random.choice(list(gates))
        a, b = random.randint(0, 1), random.randint(0, 1)
        correct = gates[name](a, b)
        guess = input(f"Round {r}: {a} {name} {b} = ? ").strip()
        if guess == str(correct):
            print("✅ Correct!")
            score += 1
        else:
            print(f"❌ Not quite — {a} {name} {b} = {correct}")
    print(f"\nFinal score: {score}/{rounds}")

# ▶ TO PLAY: delete the # on the next line, then run this cell.
# gate_arcade()

### Understanding the Code

Three ideas in this cell are worth knowing — you'll reuse all of them:

- **A dictionary of functions.** `gates` maps a name like `"AND"` to a tiny function (`lambda a, b: a & b`). A `lambda` is just a one-line, unnamed function. Looking up `gates[name]` and calling it `(a, b)` lets us pick *which* operation to run at runtime — the software version of "add gates that select the operation."
- **`&`, `|`, `^` are Python's bit operators** — they apply AND, OR, and XOR to the bits directly. On single bits they behave exactly like the truth tables above.
- **`input()` always returns a *string*.** Even if the user types `1`, you get the text `"1"`, not the number `1`. That's why we compare against `str(correct)`. Forgetting this is one of the most common beginner bugs — watch for it all semester.

### ✏️ Exercise 1 — Extend the Gate Arcade

The game above only quizzes AND, OR, and XOR. **Your task: add the NOT gate.**

- NOT takes **one** input, not two — so you'll need a separate question format for it (e.g., `NOT 1 = ?`).
- Add it as a new option the game can randomly pick.
- Play your improved game and paste your final score into a new markdown cell.

**Using AI here:** if you get stuck, ask Gemini *"how do I add a one-input NOT case to this function?"* — but before you trust its answer, check it against the NOT truth table yourself. The course rule holds: *AI is a fast first draft; you verify.*

---
## Inside the Machine: How a Computer Actually Works

We have an ALU that can do math. But where do the numbers *live*, what tells the ALU *which* math to do, and how do instructions reach the chip at all?

**Cmdr. Marlow Stack**, the lead engineer — who refers to RAM as "the good china" — explains the layout almost every computer since 1945 has used: the **von Neumann architecture**. The quick analogy is a kitchen:

- **CPU** = the cook — does the actual work, very fast, holds only a few things at once
- **Memory (RAM)** = the counter space — what you're working on *right now*; cleared when power is lost
- **Storage** (the cartridge) = the pantry — big, slow, keeps its contents unplugged
- **Input/Output** = the serving window — controller in, picture and sound out

The crucial von Neumann idea: **the program and its data live in the *same* memory.** An instruction is just a number too. Now let's open up the cook.

### Inside the CPU

A CPU is not one thing — it's a few cooperating parts:

- **Control Unit (CU)** — the traffic cop. It reads each instruction and tells every other part what to do and when. It makes no decisions of its own; it just follows the instruction.
- **ALU** — the calculator from the previous section. It only acts when the Control Unit hands it operands and an operation.
- **Registers** — a handful of ultra-fast storage slots *inside* the CPU. The named ones you should know:
  - **Program Counter (PC)** — holds the memory address of the *next* instruction. "Where am I in the program?"
  - **Instruction Register (IR)** — holds the instruction currently being carried out. "What am I doing right now?"
  - **Accumulator / general registers** — scratch space for the numbers being worked on.

When the CPU needs something from memory, it puts the address on the **address bus** and the data travels back on the **data bus** — think of buses as the hallways between the cook and the counter.

In [ ]:
# The von Neumann machine with the CPU opened up.
vn = Digraph(graph_attr={"rankdir": "LR"})
vn.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

with vn.subgraph(name="cluster_cpu") as c:
    c.attr(label="CPU", style="filled", fillcolor="oldlace")
    c.node("CU",  "Control Unit")
    c.node("ALU", "ALU")
    c.node("REG", "Registers\n(PC, IR, accumulator)")
    c.edge("CU", "ALU"); c.edge("CU", "REG"); c.edge("ALU", "REG")

vn.node("MEM", "Memory (RAM)\nprogram + data")
vn.node("IO",  "Input / Output", fillcolor="lightgreen")

vn.edge("CU", "MEM", label=" address bus ", dir="both")
vn.edge("REG", "MEM", label=" data bus ", dir="both")
vn.edge("IO", "MEM", dir="both")
vn

### Watching One Instruction Run

Suppose memory holds this tiny program. Each line is an instruction stored as a number:

| Address | Instruction | Meaning |
|---|---|---|
| 0 | `LOAD 5` | put 5 into the accumulator |
| 1 | `ADD 3` | add 3 to the accumulator |
| 2 | `STORE 9` | copy the accumulator into memory address 9 |

Here is exactly what the CPU does to run the `ADD 3` instruction (the one at address 1), step by step:

| Stage | What happens | Which part |
|---|---|---|
| **Fetch** | PC says 1 → read memory[1] → `ADD 3` lands in the IR | Control Unit, PC, IR |
| **Decode** | CU inspects the IR: "this is ADD, operand 3" | Control Unit |
| **Execute** | CU routes accumulator (5) and 3 into the ALU; ALU outputs 8; 8 goes back to the accumulator | ALU, registers |
| **Advance** | PC = 1 + 1 = 2, ready for the next instruction | Program Counter |

Every part did exactly one small job. Nothing "understood" addition — the gates from the previous section just produced 8. That is the entire trick of computing.

In [ ]:
# The same four stages, drawn as the endless loop the CPU actually runs.
fde = Digraph(graph_attr={"rankdir": "TB"})
fde.attr("node", shape="box", style="rounded,filled", fillcolor="lightyellow")

fde.node("F", "FETCH\nread instruction at PC", fillcolor="lightgreen")
fde.node("D", "DECODE\nControl Unit reads the IR")
fde.node("E", "EXECUTE\nALU / registers do the work")
fde.node("S", "ADVANCE PC\nto the next instruction", fillcolor="lightcoral")

fde.edge("F", "D"); fde.edge("D", "E"); fde.edge("E", "S")
fde.edge("S", "F", label=" repeat ")
fde

That loop is the **fetch–decode–execute cycle**, and in pseudocode it is almost insultingly simple:

```text
REPEAT FOREVER:
    1. FETCH   the instruction at the address in the Program Counter
    2. DECODE  it in the Control Unit (what operation? what operands?)
    3. EXECUTE it using the ALU and registers
    4. ADVANCE the Program Counter to the next instruction
```

Your phone runs this loop several **billion** times per second. The PixelBox 8 manages a little under two million. The *idea* is identical; only the speed changed — the same theme that ran through Notebook 1.

---
## The Memory Hierarchy

Why is there RAM *and* storage *and* registers *and* something called cache? Because no single memory technology is fast, big, and cheap at the same time. So computers use **layers**: a little very-fast memory close to the CPU, backed by progressively larger, slower, cheaper memory further away.

| Level | What it is | Speed | Typical size | Keeps data without power? |
|---|---|---|---|---|
| Registers | Slots inside the CPU | Fastest | A few dozen values | No |
| L1 / L2 cache | Tiny memory beside the cores | Very fast | KB–MB | No |
| RAM | Main working memory | Fast | GB | No |
| SSD | Flash storage | Slow-ish | Hundreds of GB | **Yes** |
| Hard disk / network | Spinning platters or remote servers | Slowest | TB and up | **Yes** |

The pattern is strict: **the closer to the CPU, the faster and more expensive per byte, and the smaller.** This is not a design flaw — it's the only way to get both speed *and* capacity at a price anyone can afford.

### How Big Are These Speed Gaps, Really?

The numbers are too small to feel. So scale them up: imagine one register access takes **1 second**. Then, very roughly:

| Memory | Real time | If a register were 1 second… |
|---|---|---|
| Register / L1 cache | ~1 ns | **1 second** |
| L2 cache | ~4 ns | ~4 seconds |
| RAM | ~100 ns | ~1.5 **minutes** |
| SSD | ~0.1 ms | ~1 **day** |
| Hard disk | ~10 ms | ~4 **months** |

Reaching all the way to a hard disk, in human terms, is the difference between answering a question now and answering it next spring. That gap is *why* games have loading screens, and why "is the data already in RAM?" is one of the most important questions in performance.

In [ ]:
import matplotlib.pyplot as plt

levels = ["Register", "L2 cache", "RAM", "SSD", "Hard disk"]
nanoseconds = [1, 4, 100, 100_000, 10_000_000]   # rough order-of-magnitude

plt.figure(figsize=(8, 4))
plt.bar(levels, nanoseconds, color="indianred")
plt.yscale("log")                                # log scale: each gap is ~10x+
plt.ylabel("access time (nanoseconds, log scale)")
plt.title("Each step away from the CPU is dramatically slower")
plt.tight_layout()
plt.show()

### Why Cache Works

Notice the y-axis is a **log scale** — each bar is many times taller than the last, so a linear chart would be unreadable. That extreme gap is the whole reason **cache** exists.

Programs have a helpful habit called **locality**: if you use a piece of data, you'll probably use it (or its neighbors) again soon. A loop runs the same instructions repeatedly; a sprite's pixels sit next to each other. So the CPU keeps recently-used data in fast cache. If the next thing it needs is there, that's a **cache hit** (fast). If not, a **cache miss** — it must trek out to RAM and wait.

"The game is loading" is exactly this: copying data from the slow pantry (disk) up to the fast counter (RAM and cache) *before* the cook needs it, so play doesn't stall mid-action.

---
## Consoles as a Fossil Record

The von Neumann architecture has barely changed since 1945. What changed is the *numbers*. Game consoles are a perfect fossil record because each generation is a fixed, well-documented snapshot of consumer hardware:

| Console | Year | CPU clock | RAM | Storage |
|---|---|---|---|---|
| Atari 2600 | 1977 | 1.2 MHz | 128 **bytes** | ROM cartridge (~4 KB) |
| NES | 1983 | 1.8 MHz | 2 KB | cartridge (~256 KB) |
| SNES | 1990 | 3.6 MHz | 128 KB | cartridge (~4 MB) |
| Nintendo 64 | 1996 | 94 MHz | 4 MB | cartridge (~32 MB) |
| PlayStation 2 | 2000 | 294 MHz | 32 MB | DVD (~4.7 GB) |
| PlayStation 5 | 2020 | ~3.5 GHz ×8 cores | 16 GB | SSD (825 GB) |

The Atari 2600 had **128 bytes** of RAM — not kilobytes, *bytes*. You could not store this sentence in it. Programmers squeezed entire games into that space by hand. The PixelBox 8 lives in roughly this era.

In [ ]:
consoles = ["Atari 2600\n1977", "NES\n1983", "SNES\n1990",
            "N64\n1996", "PS2\n2000", "PS5\n2020"]
ram_bytes = [128, 2_048, 131_072, 4_194_304, 33_554_432, 17_179_869_184]

plt.figure(figsize=(9, 4))
plt.bar(consoles, ram_bytes, color="slateblue")
plt.yscale("log")
plt.ylabel("RAM in bytes (log scale)")
plt.title("Console RAM, 1977–2020: about 130 million times bigger")
plt.tight_layout()
plt.show()

From the Atari 2600 to the PS5, RAM grew **roughly 130 million-fold**, CPU clock about 3,000-fold, and storage moved through three physical technologies (cartridge → optical disc → SSD). And yet: every one of these machines is a von Neumann computer running a fetch–decode–execute loop over the registers, ALU, and memory we just diagrammed.

**The architecture is the constant. The magnitudes are the variable.** Hold onto that — it is the single most useful idea for understanding any computer you will ever meet, from a 1977 console to whatever ships in 2040.

---
## Counting in Binary and Hexadecimal

The CPU only has transistors, and a transistor only knows two states. So every number must be written using only **two digits: 0 and 1**. This is **binary** (base 2).

You already know base 10. In `375`, each position is a power of ten: 3×100 + 7×10 + 5×1. Binary works the same way, but each position is a power of **two**. A group of 8 bits is a **byte**, with these place values:

In [ ]:
places = [128, 64, 32, 16, 8, 4, 2, 1]
byte   = [0, 1, 0, 0, 1, 0, 1, 1]            # the binary number 01001011
values = [bit * place for bit, place in zip(byte, places)]

plt.figure(figsize=(8, 4))
plt.bar([str(p) for p in places], places, color="lightgray")     # all columns
bars = plt.bar([str(p) for p in places], values, color="teal")   # 'on' columns
for bar, bit in zip(bars, byte):
    plt.text(bar.get_x() + bar.get_width()/2, 4, str(bit), ha="center")
plt.title("Binary 01001011  →  add the 'on' columns")
plt.ylabel("contribution")
plt.show()

print("Sum of the 'on' columns:", sum(values))

### Worked Example: `01001011` → decimal

Walk left to right. Wherever the bit is 1, add that column's place value:

| Bit | 0 | 1 | 0 | 0 | 1 | 0 | 1 | 1 |
|---|---|---|---|---|---|---|---|---|
| Place | 128 | 64 | 32 | 16 | 8 | 4 | 2 | 1 |
| Add? | — | 64 | — | — | 8 | — | 2 | 1 |

**64 + 8 + 2 + 1 = 75.** Going the other way (decimal → binary): repeatedly subtract the largest place value you can.

An 8-bit byte holds 0 through 255 — that's 2⁸ = 256 different values. Remember that 255. Tobble certainly does.

In [ ]:
# Python can convert for you — useful for CHECKING hand work, not replacing it.
n = 75
print(f"{n} in binary: {n:08b}")        # :08b -> binary, padded to 8 digits
print(f"{n} in hex:    {n:02X}")        # :02X -> hex, 2 digits, uppercase

from_binary = int("01001011", 2)         # the 2 means 'read this string as base 2'
print("int('01001011', 2) =", from_binary)

### Understanding the Code

Two new tools here, both worth memorizing:

- **Format specifiers inside f-strings.** `f"{n:08b}"` says "format `n` as **b**inary, at least **8** digits, zero-padded." `f"{n:02X}"` means "he**X**, 2 digits, uppercase." These are how you control how a number *prints* without changing the number itself.
- **`int(text, base)`** converts a string of digits into a number, reading it in the base you give. `int("01001011", 2)` reads that text as binary and returns `75`. `int("4B", 16)` would read hex. With no base argument, `int("75")` assumes base 10.

Use these to *check* your hand conversions, never to skip learning them — the quiz won't let you import Python.

### Hexadecimal: Binary for Humans

**Pip Renderwick**, the studio's sprite and audio artist — who can draw a dragon in 8×8 pixels and insists it's *clearly* a dragon — never writes binary by hand. She uses **hexadecimal** (base 16).

Long binary strings are error-prone for humans (`01001011` — did you miscount?). Hex fixes this because **exactly 4 bits = 1 hex digit**. Hex needs 16 digit symbols, so after 9 it borrows letters:

| Dec | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Hex | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | A | B | C | D | E | F |

So `01001011` splits into `0100` (= 4) and `1011` (= B) → **`4B`**. Two readable characters instead of eight bits. This is why colors look like `#4B2C9F` and memory addresses look like `0x1A3F`.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.4))
for d in range(16):
    ax.text(d, 1.0, f"{d:04b}", ha="center", family="monospace")
    ax.text(d, 0.6, f"{d:X}",   ha="center", family="monospace",
            color="crimson", fontsize=14)
    ax.text(d, 0.2, str(d),     ha="center", family="monospace")
ax.text(-1.6, 1.0, "binary",  ha="left")
ax.text(-1.6, 0.6, "hex",     ha="left")
ax.text(-1.6, 0.2, "decimal", ha="left")
ax.set_xlim(-1.7, 16); ax.set_ylim(0, 1.3); ax.axis("off")
ax.set_title("One reference strip: 4 bits ↔ 1 hex digit")
plt.show()

Hex isn't a *different* number system from binary in any deep sense — it's a compact, human-friendly way to write the same bits. You'll see it constantly: colors, memory addresses, error codes, file signatures. You'll drill it in the arcade shortly.

---
## Representing Negative Numbers

Tobble hits a wall. *Quest for the Reasonably Priced Sword* needs to track when the hero falls in a pit: **−50 points**. But a byte is just eight 0s and 1s. **Where does the minus sign go?**

**Naive idea — a "sign bit":** use the leftmost bit as a sign (0 = positive, 1 = negative), the other 7 bits as the size. This is *sign-magnitude*, and it almost works — but it has two ugly problems:

- There are **two zeros**: `00000000` (+0) and `10000000` (−0). Wasteful, and it breaks comparisons.
- The hardware needs special cases — the simple adder from earlier stops working.

Computers use a cleverer scheme with only one zero, where the *same* adder handles positive and negative numbers: **two's complement**.

### The Two's Complement Rule

In an 8-bit two's-complement system:

- The leftmost bit still signals sign: **0 = non-negative, 1 = negative**.
- **Positive numbers** look exactly like normal binary (`00000101` = 5).
- To get **−x**: take the binary for `x`, **flip every bit**, then **add 1**.

That's the whole rule: **invert, then add one.** The range an 8-bit byte can hold shifts to **−128 through +127** (still 256 distinct values, still one zero).

Why does it work? Because "invert and add 1" makes `x + (−x)` overflow cleanly to zero in 8 bits — so the ordinary adder just works, with no special cases. The hardware never even knows the number was "negative."

### Worked Example 1: encode **−37** as an 8-bit byte

| Step | Result |
|---|---|
| 1. Write +37 in binary | `00100101` |
| 2. Flip every bit | `11011010` |
| 3. Add 1 | `11011011` |

So **−37 = `11011011`**. The leftmost bit is 1, correctly signaling "negative."

**Check:** add `00100101` (+37) and `11011011` (−37) column by column, discard any 9th-bit carry → `00000000`. A number plus its negative is zero, exactly as it should be.

### Worked Example 2: decode `11011011` back to a signed number

The leftmost bit is **1**, so this byte is negative. Reverse the process to find the magnitude:

| Step | Result |
|---|---|
| Given byte (leftmost bit 1 → negative) | `11011011` |
| 1. Flip every bit | `00100100` |
| 2. Add 1 | `00100101` |
| 3. Read that magnitude in decimal | 37 |
| 4. Apply the negative sign | **−37** |

**Exam tip:** leftmost bit 0 → just read it as normal binary. Leftmost bit 1 → *flip and add 1* to get the magnitude, then attach a minus sign. That one rule answers every two's-complement quiz question.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(5.5, 5.5))
for raw in range(16):
    angle = np.pi/2 - raw * (2*np.pi/16)
    x, y = np.cos(angle), np.sin(angle)
    signed = raw if raw < 8 else raw - 16          # 4-bit two's complement
    ax.text(x*1.18, y*1.18, f"{raw:04b}", ha="center", va="center",
            family="monospace", fontsize=9)
    ax.text(x*0.85, y*0.85, str(signed), ha="center", va="center",
            color=("navy" if signed >= 0 else "crimson"),
            fontsize=13, fontweight="bold")
ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, color="gray"))
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("4-bit two's complement: bits (outer) vs. signed value (inner)")
plt.show()

On the wheel, as the bits climb past the halfway point (`1000`) the *meaning* jumps to the most negative value and climbs back toward −1 — the numbers wrap like an odometer. Python's own integers don't have this limit (they grow as large as needed), but **fixed-width** values — registers, image pixels, the PixelBox 8's score counter — absolutely do. We can simulate a fixed 8-bit value with a *mask*:

In [ ]:
value = -37
byte  = value & 0xFF          # keep only the lowest 8 bits
print(f"{value} stored in 8 bits: {byte:08b}  (raw value {byte})")

raw = 0b11011011
signed = raw - 256 if raw >= 128 else raw   # leftmost bit set -> negative
print("11011011 interpreted as signed:", signed)

### Understanding the Code

- **`& 0xFF` is a mask.** `0xFF` is hex for `11111111` — eight 1s. ANDing any number with it keeps only the lowest 8 bits and throws everything else away. This is *exactly* what a real 8-bit register does to a value that's too big: it simply has nowhere to put the higher bits.
- **`0b11011011` is a binary literal.** The `0b` prefix tells Python "the following digits are binary." (`0x` does the same for hex.) It's just another way to write the number `219`.
- **The signed-read line** (`raw - 256 if raw >= 128 else raw`) is the two's-complement decode rule in one line: if the value is 128 or more, the leftmost bit is set, so it really represents `raw - 256` (a negative number). This is the code form of the exam tip above.

Here is the notebook's main test-prep tool. Run it, read the walk-through, then do the exercise underneath.

In [ ]:
def base_arcade(mode="bin2dec", rounds=5):
    """Random number-base practice. Modes: bin2dec, dec2bin, hex, twos."""
    score = 0
    for r in range(1, rounds + 1):
        if mode == "bin2dec":
            n = random.randint(0, 255)
            prompt, correct = f"Round {r}: binary {n:08b} = ? (decimal) ", str(n)
        elif mode == "dec2bin":
            n = random.randint(0, 255)
            prompt, correct = f"Round {r}: decimal {n} = ? (8-bit binary) ", f"{n:08b}"
        elif mode == "hex":
            n = random.randint(0, 255)
            prompt, correct = f"Round {r}: decimal {n} = ? (2-digit hex) ", f"{n:02X}"
        elif mode == "twos":
            n = random.randint(-128, 127)
            prompt, correct = f"Round {r}: {n} as an 8-bit two's-complement byte = ? ", f"{n & 0xFF:08b}"
        else:
            print("Unknown mode. Use bin2dec, dec2bin, hex, or twos.")
            return
        guess = input(prompt).strip().upper()
        if guess == correct:
            print("✅ Correct!")
            score += 1
        else:
            print(f"❌ Answer was {correct}")
    print(f"\n[{mode}] score: {score}/{rounds}")

# ▶ TO PLAY: delete the # on ONE line below, then run this cell.
# base_arcade("bin2dec")
# base_arcade("dec2bin")
# base_arcade("hex")
# base_arcade("twos")

### Understanding the Code

- **The `if / elif` chain is a dispatcher.** The `mode` string selects which kind of question to build. Each branch sets two things: the `prompt` to show, and the `correct` answer as a *string* (so it can be compared to what `input()` returns).
- **The format specifiers do the heavy lifting.** `f"{n:08b}"`, `f"{n:02X}"`, and `f"{n & 0xFF:08b}"` are the same tools from earlier — note the `twos` mode reuses the `& 0xFF` mask to produce the two's-complement byte automatically.
- **`.strip().upper()` normalizes the answer.** `.strip()` removes accidental spaces; `.upper()` makes hex like `4b` match `4B`. Cleaning user input before comparing it is a habit worth building now.

### ✏️ Exercise 2 — Extend the Number Base Arcade

**Self-study:** play every mode (`bin2dec`, `dec2bin`, `hex`, `twos`) until you can hit a streak of 5. This is your quiz trainer — replay it as much as you want.

**Coding task:** add a **`"hex2bin"`** mode that shows a 2-digit hex value and asks for the 8-bit binary.

- You'll add one more `elif mode == "hex2bin":` branch.
- It needs a `prompt` (show the hex) and a `correct` answer (the binary string).
- Test it against the reference strip from earlier in the notebook.

**Using AI here:** ask Gemini *"write just the elif branch for a hex2bin mode matching this function's style"* — then **run it and check several answers by hand** before trusting it. Paste your scores and describe your change in a markdown cell.

---
## Encoding Text, Images, and Sound

Pip Renderwick's rule for the whole studio: **"if it's in the game, it's a number somewhere."** Numbers were the easy part — they *are* numbers. The trick is turning everything else into numbers too.

### Text = Numbers in a Lookup Table

A computer stores the letter `A` by agreeing, in advance, that `A` *is* the number 65. That agreement is a **character encoding**. The classic one is **ASCII**: `A` = 65, `B` = 66, `a` = 97, `space` = 32, and so on.

You met this in Notebook 1's Caesar cipher — `ord()` gives a character's code, `chr()` turns a code back into a character. ASCII only covers 128 characters (enough for English). Modern systems use **Unicode**, extending the same idea to every writing system on Earth plus emoji — `🐉` is just a (large) number with an agreed meaning.

In [ ]:
codes = np.arange(32, 128).reshape(8, 12)     # printable ASCII as an 8x12 grid
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(codes, cmap="viridis")
for i in range(8):
    for j in range(12):
        c = int(codes[i, j])
        ax.text(j, i, f"{chr(c)}\n{c}", ha="center", va="center",
                color="white", fontsize=8)
ax.set_title("Printable ASCII: every character is just a number")
ax.axis("off")
plt.show()

**Understanding the code:** `np.arange(32, 128)` builds the list of numbers 32–127; `.reshape(8, 12)` folds that flat list into an 8-row by 12-column grid (8 × 12 = 96 numbers). `imshow` paints any grid of numbers as a colored image — we'll reuse it immediately for sprites. The `chr(c)` call turns each code back into its character so we can label it. One idea, many uses: *a grid of numbers is a picture.*

### Images = Grids of Numbers

A digital image is a grid of **pixels**, and each pixel is a number. The simplest case: 1 bit per pixel — 0 is background, 1 is drawn. That's exactly how a PixelBox 8 sprite works. Color just means more bits per pixel plus a small **palette** (Vesper only paid for four colors). The image below is rendered straight from a grid of 0s and 1s.

In [ ]:
# Pip's 8x8 'definitely a dragon' sprite — just a grid of bits.
dragon = [
    [0, 0, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0],
    [1, 1, 0, 1, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 0, 0, 1, 1, 0],
    [1, 1, 0, 0, 0, 0, 1, 1],
]

plt.figure(figsize=(3, 3))
plt.imshow(dragon, cmap="binary")   # same imshow as the ASCII grid
plt.title("8 bytes = one sprite")
plt.axis("off")
plt.show()

### Sound = Numbers Over Time

Sound is a wave. To store it, the computer measures the wave's height many times per second; each measurement is a number — that's **sampling**, and the stored sound is literally just that list of numbers. The PixelBox 8's audio chip is cheap, so instead of smooth waves it mostly produces blocky **square waves** — the classic "chiptune" buzz.

In [ ]:
t = np.linspace(0, 1, 600)            # 600 evenly spaced sample times
sine   = np.sin(2 * np.pi * 4 * t)
square = np.sign(sine)               # np.sign flattens the wave to -1 / +1

plt.figure(figsize=(9, 3))
plt.plot(t, sine,   label="smooth wave (real world)")
plt.plot(t, square, label="square wave (PixelBox 8 audio)")
plt.legend(loc="upper right")
plt.title("A sound, stored as a list of numbers over time")
plt.yticks([-1, 0, 1])
plt.show()

**Understanding the code:** `np.linspace(0, 1, 600)` makes 600 evenly spaced time points — those are the sample moments. `np.sign(...)` snaps every value to −1 or +1, which is exactly what a cheap 1-bit audio channel does to a wave.

Text, image, sound: three completely different experiences, **one underlying idea** — a sequence of numbers plus an agreed rule for what those numbers mean. Change the rule and the same bytes become gibberish. That's the bridge to the final part.

### ✏️ Exercise 3 — Design Your Own Sprite

The next cell is **yours to edit directly**. Run it first to see the starter shape, then read the task below it.

In [ ]:
# ✏️ EDIT THESE 0s and 1s to draw your own 8x8 sprite.
frame1 = [
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
]

# Optional second frame for the animation bonus (start as a copy, then tweak):
frame2 = [
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 0, 0, 0],
]

plt.figure(figsize=(3, 3))
plt.imshow(frame1, cmap="binary")
plt.title("Your sprite")
plt.axis("off")
plt.show()

**Your task:** edit the `0`s and `1`s in `frame1` above to draw something for *Quest for the Reasonably Priced Sword* — a sword, a coin, a slime, anything. A `1` is a filled pixel, a `0` is empty. Re-run the cell to see your art.

**Optional bonus — animation:** fill in `frame2` with a slightly different pose, then run the cell below to flip between them. Two frames is exactly how 1980s games made things "move" on a tiny memory budget.

**Using AI here (optional):** ask Gemini *"give me an 8×8 grid of 0s and 1s that looks like a sword"* — then paste it in and judge whether it actually does. You are the art director; the AI is a junior intern.

In [ ]:
# ▶ ANIMATION BONUS: run this after filling in frame2.
import time
from IPython.display import clear_output

for _ in range(6):                       # 6 flips, then stop
    for frame, label in [(frame1, "frame 1"), (frame2, "frame 2")]:
        clear_output(wait=True)
        plt.figure(figsize=(3, 3))
        plt.imshow(frame, cmap="binary")
        plt.title(label)
        plt.axis("off")
        plt.show()
        time.sleep(0.3)

---
## When Representation Breaks

**Quoth Bugbear**, the studio's QA tester — who finds the bug, was right all along, and is insufferable about it — files report #256:

> *"At level 256, the game corrupts. Enemies turn into garbage tiles. Half the screen is hex soup. Unplayable."*

The level counter is stored in **one byte**. A byte holds 0–255. When the game adds 1 to level 255, the value doesn't become 256 — there's no room for a 9th bit, so it **wraps around to 0** (or worse, corrupts neighboring memory). This is **integer overflow**, and it is not a fictional problem.

### The Real One: the Pac-Man Kill Screen

The 1980 arcade game **Pac-Man** stored its level number in a single byte and used it to draw the bonus-fruit display. At **level 256** that byte overflowed; the draw routine ran wild and corrupted the right half of the screen into a famous garbled mess — the "**kill screen**." The game was literally unbeatable past that point, not by design, but because of one undersized integer.

| Decision | Consequence |
|---|---|
| Store level in 1 byte | Saves memory (precious in 1980) |
| Never expect 256 levels | Reasonable — almost no one got there |
| No overflow check | Level 256 corrupts memory → kill screen |

The same bug class still bites: a 2015 FAA directive warned that a Boeing 787 counter could overflow after 248 days of continuous power, potentially shutting down electrical generators in flight. **Representation size is a safety decision.**

In [ ]:
# Simulate an 8-bit level counter. & 0xFF forces it to stay 8 bits wide.
level = 253
for _ in range(6):
    level = (level + 1) & 0xFF
    note = "  <-- overflow! wrapped around" if level == 0 else ""
    print(f"next level -> stored as {level:3d}  ({level:08b}){note}")

Run that cell and watch 254 → 255 → **0**. Nothing is "broken" — the hardware did exactly what fixed-width wrap-around always does (the same `& 0xFF` masking from the two's-complement section). The bug was a *human* decision: choosing a box too small for the numbers that would eventually go in it.

Every representation choice in this notebook — byte width, palette size, sample rate, character encoding — is an engineering trade-off with real consequences. That is the deep lesson of data representation, and it's why this is Learning Outcomes 1 and 2 of the course, not an afterthought.

### ✏️ Exercise 4 — Build a Retro Toy with AI

Use Gemini (or Claude / ChatGPT) to build **one small, fun thing** that runs in this notebook and uses a concept from it. Pick one:

| Option | What it does | Concept it uses |
|---|---|---|
| **Secret Message Machine** | Turns text → binary or hex, and back again | `ord`/`chr`, binary/hex |
| **Pixel Banner** | Turns your initials into blocky 8-bit letters with `imshow` | bitmaps / sprite grids |
| **Chiptune Plotter** | Turns a few notes into a plotted square-wave "melody" | sound = numbers, square waves |
| **Overflow Survival** | A tiny text game where your score is an 8-bit byte and wraps at 256 | two's complement / overflow |

**Steps:**
1. Ask the AI to draft your chosen toy. Be specific: *"...as a single Python cell that runs in Google Colab."*
2. Paste it into the empty cell below and get it actually working (it probably won't run perfectly first try).
3. In a markdown cell, write 2–3 sentences: **what did the AI get wrong or what did you improve, and how did you fix it?**

Remember the course rule: *AI is a fast first draft. You verify.*

In [ ]:
# ✏️ Paste your AI-built retro toy here, then run it and fix what's broken.


---
## Key Terms

- **Transistor** — A microscopic electrical switch with two states (on/off); the physical basis of all digital computing.
- **Logic gate** — A small group of transistors that follows one logical rule (AND, OR, NOT, XOR).
- **Truth table** — A table listing every input combination for a gate or circuit and the resulting output.
- **Half-adder** — An XOR gate plus an AND gate that together add two bits (XOR = sum, AND = carry).
- **ALU (Arithmetic Logic Unit)** — The CPU part that performs arithmetic and logic, built from chained adders and gates.
- **Control Unit** — The CPU part that reads each instruction and directs all other parts.
- **Register** — A tiny, extremely fast storage slot inside the CPU.
- **Program Counter (PC)** — The register holding the address of the next instruction.
- **Instruction Register (IR)** — The register holding the instruction currently being executed.
- **Bus** — The wiring that carries addresses or data between the CPU and memory.
- **von Neumann architecture** — The standard design where CPU, memory, and I/O are connected and program instructions live in the same memory as data.
- **Fetch–decode–execute cycle** — The repeating loop a CPU performs to run a program.
- **Memory hierarchy** — The layered arrangement of registers, cache, RAM, and storage trading speed against size and cost.
- **Cache** — Small fast memory near the CPU that holds recently used data; a **hit** is found there, a **miss** is not.
- **Locality** — The tendency of programs to reuse recent data and nearby data, which makes caching effective.
- **Bit / Byte** — A bit is one binary digit; a byte is 8 bits (256 possible values).
- **Binary (base 2)** — A number system using only 0 and 1; place values are powers of two.
- **Hexadecimal (base 16)** — Compact notation where 4 bits map to one hex digit (0–9, A–F).
- **Two's complement** — The standard scheme for storing signed integers: negate by flipping all bits and adding 1.
- **Integer overflow** — When a value exceeds the maximum its fixed-size storage can hold and wraps around.
- **Character encoding (ASCII / Unicode)** — An agreed mapping from characters to numbers.
- **Pixel** — One dot of an image, stored as a number; more bits per pixel means more possible colors.
- **Sampling** — Measuring a sound wave's height many times per second to store it as a list of numbers.
- **Palette** — A small fixed set of colors an image is allowed to use, to save memory.

---
## Summary

- **Transistors → gates → half-adders → ALU.** Computation is switches wired to follow logical rules.
- **Inside the CPU**, a Control Unit directs an ALU and registers (PC, IR, accumulator) through the **fetch–decode–execute** loop, talking to memory over buses.
- **The memory hierarchy** (registers → cache → RAM → SSD → disk) exists because no memory is fast, big, and cheap at once; **caching** works because programs have locality.
- **Consoles from 1977 to 2020** show RAM growing ~130 million-fold while the architecture stayed the same — magnitudes change, ideas don't.
- **Binary** is counting in powers of two; **hexadecimal** is the same bits, written for humans.
- **Two's complement** stores negatives so the ordinary adder still works — *flip the bits, add one*.
- **Text, images, and sound** are all numbers plus an agreed rule for reading them.
- **Representation size is an engineering and safety decision** — the Pac-Man kill screen was one byte, chosen too small.

Everything is numbers. The size of the box you put them in matters.

---
## What's Next

You now know what the machine *is*, how it computes, and how it stores things. **Notebook 3** turns to instructing it: we'll go from a plain-English description of a problem, to **pseudocode**, to a **flowchart**, to working **Python** — the authoring workflow this whole course runs on. Eight Bits & Bob still has a game to finish, and the code isn't going to write itself. (Bob keeps insisting it will. Nobody listens to Bob.)

---
*COMP 1150 — Computer Science Concepts · Brendan Shea, PhD*  
*Content licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).*